# Milestone 9c - Analyse and decide

Part 3 of 4: the **Evaluate** stage. In 9b you ran four versions on the dev
questions, twice each: **v1** (in production), **v2** (rewrite + rerank, the
change), **v2-no-rewrite** (the ablation) and **oracle** (perfect retrieval).

Now you analyse them in a set order. The order matters: each step can stop the
process or change how you read the next one.

1. **Must-pass gate**: did v2 break anything critical?
2. **Oracle**: how much could better retrieval possibly gain?
3. **Groups**: where are the differences?
4. **Flaky questions**: which results are just noise?
5. **Paired test**: is v2's win real?
6. **Read the regressions and improvements**: why did it win or lose?
7. **The reranker check**: what did the reranker actually do?
8. **Ablation**: is the rewrite worth its cost?
9. **Time guardrail**: how much slower is it?
10. **Decide** by the rule written in 9b, then **confirm on the holdout**.

In [ ]:
import ipytest
import mlflow
import pandas as pd
import pytest
from mlflow.entities import SpanType

from drill import rag_lab as R

ipytest.autoconfig(clean="test_*")

URI = R.use_mlflow("m9-help-center")
SPLIT = R.default_split()
DEV = [q for q in R.QUESTIONS if SPLIT[q["id"]] == "dev"]
HOLDOUT = [q for q in R.QUESTIONS if SPLIT[q["id"]] == "holdout"]
VARIANTS = ["v1", "v2", "v2-no-rewrite", "oracle"]

saved = R.load_results()
if saved:
    results, SESSION, source = saved["results"], saved["session"], "your live run from 9b"
    lab = R.Lab()
else:   # replay the recorded experiment into MLflow, so its traces are here to inspect
    lab = R.Lab(live=False)
    results, SESSION, source = lab.run(DEV, VARIANTS, runs=2, tags={"stage": "experiment"}), lab.session, "the recording"

df = pd.DataFrame(results)
PASSED = {(v, r): dict(zip(g["qid"], g["passed"])) for (v, r), g in df.groupby(["variant", "run"])}
print(f"Analysing {len(df)} requests from {source}.")

## 1. The must-pass gate comes first

If v2 breaks any must-pass question that v1 gets right, it doesn't ship, whatever
else it does. So check that before looking at anything else. A must-pass question
has to pass on **both** runs: passing only sometimes isn't passing.

The gate also shows something useful about production: any must-pass question
that **v1 already fails** is a live problem, whatever the experiment decides.

In [ ]:
def holds(variant, qid):
    return PASSED[(variant, 0)][qid] and PASSED[(variant, 1)][qid]


def gate(candidate):
    regressed = [q for q in R.MUST_PASS if holds("v1", q) and not holds(candidate, q)]
    already_failing = [q for q in R.MUST_PASS if not holds("v1", q)]
    return regressed, already_failing


regressed, already = gate("v2")
print("already failing in production (v1):", [(q, R.BY_ID[q]["question"]) for q in already])
print("v2 breaks:", [(q, R.BY_ID[q]["question"]) for q in regressed] or "nothing")

**What happened.** On the recorded run the gate finds two things, and both
matter more than any average.

- **v2 breaks a must-pass question**: "When will my money show up after I return
  an item?" v1 gets it right; v2 doesn't. By the rule, v2 can't ship, whatever
  the rest of the analysis says. Keep going anyway: the rest explains *why*, and
  points at what to try instead.
- **v1 already fails one in production**: "How long do I have to send something
  back?" Nobody knew until the must-pass set was written down. That's a live
  problem to fix, experiment or not.

## 2. The oracle: how much can retrieval gain at all?

The oracle got the right article handed to it. Its score is a ceiling for any
change to retrieval: no reranker can beat perfect retrieval. The gap between v1
and the oracle is retrieval's **headroom**.

In [ ]:
def rate(variant):
    both = list(PASSED[(variant, 0)].values()) + list(PASSED[(variant, 1)].values())
    return sum(both) / len(both)


for variant in VARIANTS:
    print(f"{variant:15} {rate(variant):6.1%}")
print(f"\nretrieval headroom (oracle - v1): {round((rate('oracle') - rate('v1')) * 100)} points")
print(f"how much of it v2 captured:        {round((rate('v2') - rate('v1')) * 100)} points")

**What happened.** With perfect retrieval the app gets about 90% right, against
about 73% for v1. So retrieval has roughly **18 points of headroom**, and v2
captures about 11 of them. Retrieval is worth working on.

The oracle also says what retrieval *can't* fix: even with the right article,
three questions still fail. Those are answer problems (or scorer problems), and no
retrieval change will help them.

## 3. By group, with ranges

The overall rate can hide a group getting worse. Look at every group, with a
Wilson range from milestone 8's helper idea, and the traffic-weighted total.

In [ ]:
def wilson(passed, total, z=1.96):
    if total == 0:
        return (0.0, 1.0)
    p = passed / total
    d = 1 + z * z / total
    c = (p + z * z / (2 * total)) / d
    h = z * ((p * (1 - p) / total + z * z / (4 * total * total)) ** 0.5) / d
    return (max(0.0, c - h), min(1.0, c + h))


rows = []
for group in R.TRAFFIC:
    ids = [q["id"] for q in DEV if q["group"] == group]
    row = {"group": group, "questions": len(ids)}
    for variant in ["v1", "v2", "oracle"]:
        p = sum(PASSED[(variant, 0)][i] for i in ids)
        lo, hi = wilson(p, len(ids))
        row[variant] = f"{p}/{len(ids)} ({lo:.0%}-{hi:.0%})"
    rows.append(row)
display(pd.DataFrame(rows).set_index("group"))

for variant in ["v1", "v2"]:
    by_group = {g: sum(PASSED[(variant, 0)][q["id"]] for q in DEV if q["group"] == g)
                   / sum(1 for q in DEV if q["group"] == g) for g in R.TRAFFIC}
    weighted = sum(by_group[g] * R.TRAFFIC[g] for g in by_group) / sum(R.TRAFFIC.values())
    print(f"{variant}: weighted by real traffic {weighted:.1%}")

**What happened.** Almost all of v2's gain is in **billing** (5 of 9 to 8 of 9).
Shipping doesn't move, fraud and unanswerable questions were already perfect, and
**refunds** stay the weakest group even with perfect retrieval. The ranges are
wide, because each group has only 4 to 9 dev questions: these are directions to
look in, not proof.

## 4. Flaky questions: separate noise from evidence

Each version ran twice. A question that passed on one run and failed on the
other is **flaky**: its result is partly luck. If it also differs between v1 and
v2, that difference may be luck too. Set flaky questions aside before the paired
test.

**Your turn.** Write `flaky(run0, run1)`: two dicts from question id to passed,
for the same version. Return the sorted list of ids whose result differs.

In [ ]:
def flaky(run0, run1):
    """Sorted ids whose result differs between two runs of the same version."""
    # TODO: compare run0 and run1 for each id; keep the ids that differ, sorted.
    raise NotImplementedError("Milestone 9c: implement flaky")

In [ ]:
%%ipytest -qq

def test_finds_the_differences():
    assert flaky({"a": True, "b": False, "c": True}, {"a": False, "b": False, "c": True}) == ["a"]


def test_sorted():
    assert flaky({"z": True, "m": False}, {"z": False, "m": True}) == ["m", "z"]


def test_identical_runs():
    assert flaky({"a": True}, {"a": True}) == []

In [ ]:
FLAKY = {}
for variant in VARIANTS:
    FLAKY[variant] = flaky(PASSED[(variant, 0)], PASSED[(variant, 1)])
    print(f"{variant:15} {len(FLAKY[variant])} flaky: {FLAKY[variant]}")
SET_ASIDE = sorted(set(FLAKY["v1"]) | set(FLAKY["v2"]))
print("\nset aside for the v1-v2 comparison:", SET_ASIDE)

**What happened.** v1 and v2 gave the same pass or fail on both runs for every
question, so nothing needs setting aside for their comparison. The oracle has one
flaky question. With more randomness (a higher temperature, a bigger model, an
LLM judge) expect more, and expect them to matter.

## 5. The paired test: is v2's win real?

Compare v1 and v2 question by question, on the questions that weren't flaky.
Only the ones where they disagree count: **improvements** (v2 right, v1 wrong)
and **regressions** (the reverse).

**Your turn.** Write `changes(old, new)`: two dicts from question id to passed.
Return `{"improvements": [...], "regressions": [...]}`, each a sorted list of ids.

In [ ]:
def changes(old, new):
    """{"improvements": sorted ids new got right and old didn't, "regressions": the reverse}."""
    # TODO: walk the ids; an improvement is new right and old wrong, a regression the reverse.
    raise NotImplementedError("Milestone 9c: implement changes")

In [ ]:
%%ipytest -qq

def test_both_directions():
    old = {"a": False, "b": True, "c": True, "d": False}
    new = {"a": True, "b": False, "c": True, "d": False}
    assert changes(old, new) == {"improvements": ["a"], "regressions": ["b"]}


def test_nothing_changed():
    assert changes({"a": True}, {"a": True}) == {"improvements": [], "regressions": []}

In [ ]:
def mcnemar(improvements, regressions):
    n, gap = improvements + regressions, improvements - regressions
    return "too close to call" if n == 0 or gap * gap < 4 * n else ("the new version wins" if gap > 0 else "v1 wins")


old = {q: ok for q, ok in PASSED[("v1", 0)].items() if q not in SET_ASIDE}
new = {q: ok for q, ok in PASSED[("v2", 0)].items() if q not in SET_ASIDE}
CHANGES = changes(old, new)
i, r = len(CHANGES["improvements"]), len(CHANGES["regressions"])
print(f"improvements {i}: {CHANGES['improvements']}")
print(f"regressions  {r}: {CHANGES['regressions']}")
print(f"McNemar: gap {i - r}, needs at least 2 x sqrt({i + r}) = {2 * (i + r) ** 0.5:.1f}  ->  {mcnemar(i, r)}")

**What happened.** 6 improvements against 2 regressions. A 6-to-2 split *looks*
like a win, but McNemar's rule needs a gap of about 5.7 and the gap is 4: **too
close to call**. Combined with the gate failure, v2 is not shipping. Now find out
why.

## 6. Read the regressions and improvements

The count says *whether*. Reading says *why*. For each changed question, line up
what v1's search found with what v2's reranker passed on, using the traces'
**RETRIEVER** and **RERANKER** steps.

In [ ]:
traces = mlflow.search_traces(filter_string=f"tags.session = '{SESSION}' AND tags.run = '0'", return_type="list")
TRACE = {(t.info.tags["variant"], t.info.tags["qid"]): t for t in traces}


def docs_sent(variant, qid):
    """The articles that reached the answer step, read from the trace."""
    trace = TRACE[(variant, qid)]
    steps = trace.search_spans(span_type=SpanType.RERANKER) or trace.search_spans(span_type=SpanType.RETRIEVER)
    return [d["doc_id"] for d in steps[0].outputs]


for kind in ["improvements", "regressions"]:
    print(kind.upper())
    for qid in CHANGES[kind]:
        q = R.BY_ID[qid]
        print(f"  {qid:12} right article: {q['gold'] or '(none: unanswerable)'}")
        print(f"     {q['question']}")
        print(f"     v1 sent {docs_sent('v1', qid)}   v2 sent {docs_sent('v2', qid)}")

Read each one: the question, the right article, what each version sent to the
answer step, and the answers themselves in the MLflow UI. Then write the cause in
a few words. These are the causes for the recorded run.

In [ ]:
CAUSES = {
    "billing-06": "the reranker moved the right article into the top 3",
    "billing-08": "the reranker moved the right article into the top 3",
    "billing-12": "the reranker moved the right article into the top 3",
    "shipping-01": "the reranker moved the right article into the top 3",
    "refund-01": "the rewrite found an article keyword search missed",
    "refund-07": "the rewrite found an article keyword search missed",
    "refund-02": "the rewrite pushed the right article out of the search results",
    "shipping-07": "the scorer missed a correct wording ('before your order ships')",
}
for kind in ["improvements", "regressions"]:
    counts = {}
    for qid in CHANGES[kind]:
        cause = CAUSES.get(qid, "unlabelled")
        counts[cause] = counts.get(cause, 0) + 1
    print(kind, counts)

**What happened.** Reading them turned a "too close to call" into three specific
findings:

1. **The reranker works.** Four improvements come from it moving the right
   article into the top 3, and it caused none of the regressions.
2. **The rewrite cuts both ways.** It found two articles keyword search missed
   ("send something back" now finds the return-window article), but it also
   reworded "when will my money show up" so badly that the refund-timing
   article dropped out of the search results. That's the must-pass regression.
3. **One "regression" isn't one.** v2 answered "you can change it... before your
   order ships", which is correct, but the expected facts only accepted "until it
   ships" and similar. The **scorer** was wrong, not the app. This is why you read
   regressions instead of trusting the count.

In [ ]:
# Fix the expected facts for shipping-07 (9d writes the fix into the eval set too), then re-score.
R.BY_ID["shipping-07"]["facts"] = ["until the order ships | before it ships | until it ships | hasn't shipped | "
                                   "has not shipped | until your order ships | before your order ships"]
for (variant, run), passed in PASSED.items():
    one = [x for x in results if x["variant"] == variant and x["run"] == run and x["qid"] == "shipping-07"]
    if one:
        passed["shipping-07"] = R.score(one[0]["answer"], R.BY_ID["shipping-07"])
CHANGES = changes({q: ok for q, ok in PASSED[("v1", 0)].items() if q not in SET_ASIDE},
                  {q: ok for q, ok in PASSED[("v2", 0)].items() if q not in SET_ASIDE})
print("after fixing the scorer:", {k: len(v) for k, v in CHANGES.items()},
      "->", mcnemar(len(CHANGES["improvements"]), len(CHANGES["regressions"])))

**What happened.** With the scorer fixed it's 6 improvements to 1 regression:
still too close (it needs a gap of about 5.3), and the must-pass regression is
still there. Fixing the measurement doesn't rescue v2, but it does make every
number after this one honest.

## 7. What did the reranker actually do?

For every dev question, compare the search's own top 3 with the reranker's top
3, on the same ten candidates (v2-no-rewrite isolates the reranker: same search
as v1, plus the reranker). Four outcomes: **helped** (the right article moved
into the top 3), **hurt** (moved out), **no change**, and **out of reach** (not
among the ten at all, so no reranker could help).

In [ ]:
rr = mlflow.search_traces(filter_string=f"tags.session = '{SESSION}' AND tags.variant = 'v2-no-rewrite' AND tags.run = '0'", return_type="list")
outcomes = {"helped": [], "hurt": [], "no_change": [], "out_of_reach": []}
for t in rr:
    gold = R.BY_ID[t.info.tags["qid"]]["gold"]
    if not gold:
        continue
    before = [d["doc_id"] for d in t.search_spans(span_type=SpanType.RETRIEVER)[0].outputs]
    after = [d["doc_id"] for d in t.search_spans(span_type=SpanType.RERANKER)[0].outputs]
    if gold not in before:
        outcomes["out_of_reach"].append(t.info.tags["qid"])
    elif gold in after[:3] and gold not in before[:3]:
        outcomes["helped"].append(t.info.tags["qid"])
    elif gold in before[:3] and gold not in after[:3]:
        outcomes["hurt"].append(t.info.tags["qid"])
    else:
        outcomes["no_change"].append(t.info.tags["qid"])
for k, v in outcomes.items():
    print(f"{k:13} {len(v):2}  {sorted(v)}")

**What happened.** On its own, the reranker **helped 6 questions and hurt none**.
Two questions were **out of reach**: their right article wasn't among the ten
candidates, so no reranker could help. Those are exactly the two the rewrite
rescued. So the reranker is safe and useful; the rewrite is the risky part.

## 8. The ablation: what does the rewrite add?

v2-no-rewrite is v2 with the rewrite removed. The difference between them is
what the rewrite contributes: paired, on the same questions.

In [ ]:
ab = changes({q: ok for q, ok in PASSED[("v2-no-rewrite", 0)].items() if q not in SET_ASIDE},
             {q: ok for q, ok in PASSED[("v2", 0)].items() if q not in SET_ASIDE})
print(f"v2-no-rewrite {rate('v2-no-rewrite'):.1%}   v2 {rate('v2'):.1%}")
print(f"adding the rewrite: {len(ab['improvements'])} improvements {ab['improvements']}, "
      f"{len(ab['regressions'])} regressions {ab['regressions']}  ->  {mcnemar(len(ab['improvements']), len(ab['regressions']))}")

**What happened.** Adding the rewrite to the reranker gives 2 improvements (the two
out-of-reach questions) and 1 regression (the refund-timing one; with the scorer
fixed, the shipping question is no longer counted). That's too close to call, it
adds about 0.3 seconds, and its one regression is a must-pass question. As built,
the rewrite isn't worth it.

## 9. The time guardrail

Every step's trace records how long its model call took (when replaying, the
time it took when it was recorded). Add them up per request.

In [ ]:
everything = mlflow.search_traces(filter_string=f"tags.session = '{SESSION}'", return_type="list")
ms = {}
for t in everything:
    ms.setdefault(t.info.tags["variant"], []).append(R.model_ms(t))
TIME = {v: sorted(xs)[len(xs) // 2] for v, xs in ms.items()}
for v in VARIANTS:
    print(f"{v:15} median {TIME[v] / 1000:5.1f} s of model time per request")
print(f"\nv2 adds {(TIME['v2'] - TIME['v1']) / 1000:.1f} s at the median; the guardrail allows 3.0 s")

**What happened.** v1 needs about 1.1 seconds of model time per request; v2 about
3.3 (a rewrite, a rerank and the answer). That's 2.2 seconds more, inside the
3-second guardrail, but users will notice it. The reranker alone, without the
rewrite, is about 3.0.

## 10. Decide by the rule

The rule from 9b: *ship v2 only if no must-pass question regresses against v1,
it wins clearly by McNemar's rule on dev, it stays within the time guardrail, and
the win holds on the holdout.* Apply it exactly as written.

In [ ]:
def decide(candidate, open_holdout):
    regressed, _ = gate(candidate)
    ch = changes({q: ok for q, ok in PASSED[("v1", 0)].items() if q not in SET_ASIDE},
                 {q: ok for q, ok in PASSED[(candidate, 0)].items() if q not in SET_ASIDE})
    verdict = mcnemar(len(ch["improvements"]), len(ch["regressions"]))
    extra = TIME[candidate] - TIME["v1"]
    print(f"{candidate}: must-pass regressions {regressed or 'none'} | dev {len(ch['improvements'])} improvements, "
          f"{len(ch['regressions'])} regressions -> {verdict} | +{extra / 1000:.1f} s")
    if regressed or verdict != "the new version wins" or extra > 3000:
        return False
    if not open_holdout:
        return True
    hold = (lab if saved else R.Lab(live=False)).run(HOLDOUT, ["v1", candidate], runs=1, tags={"stage": "holdout"})
    h = {v: {x["qid"]: x["passed"] for x in hold if x["variant"] == v} for v in ["v1", candidate]}
    hc = changes(h["v1"], h[candidate])
    print(f"  holdout ({len(HOLDOUT)} questions, opened once): {len(hc['improvements'])} improvements {hc['improvements']}, "
          f"{len(hc['regressions'])} regressions {hc['regressions']}")
    return not hc["regressions"] and len(hc["improvements"]) >= 1


print("DECISION:", "ship v2" if decide("v2", open_holdout=True) else "keep v1")

**What happened.** v2 fails the rule at the first check, the must-pass
regression, so the holdout is never opened for it. **Keep v1.**

But the analysis points clearly at a different version: **the reranker without
the rewrite** (v2-no-rewrite). It helped six questions, hurt none, and didn't
touch the refund-timing question.

**Careful.** You picked it *after* looking at the results. That's exactly the
"testing many variants" trap from the Daily Drill: its dev numbers are now a
little optimistic. So treat it as a new, planned experiment with the same rule
and **one** comparison, and lean on the holdout, which is still untouched because
v2 never got that far.

In [ ]:
SHIP = "v2-no-rewrite" if decide("v2-no-rewrite", open_holdout=True) else "v1"
print("FOLLOW-UP DECISION:", "ship " + SHIP if SHIP != "v1" else "keep v1")

**What happened.** The reranker-only version breaks no must-pass question (it
still fails "How long do I have to send something back?", but v1 already did, so
that's not a regression). It wins on dev, 4 improvements to 0 regressions, which
just meets McNemar's rule. It adds about 1.9 seconds. And on the holdout, opened
once, it gets one more question right and none wrong. **Ship the reranker-only
version.**

What you learned that a single score never would have told you:

| Finding | Where it came from | What to do with it |
|---|---|---|
| v1 was already failing a must-pass question | the gate | fix it next, as its own experiment |
| retrieval has about 18 points of headroom | the oracle | keep investing in retrieval |
| the reranker helps and doesn't hurt | reading the improvements, the reranker check | ship it |
| the rewrite helps two questions but breaks a critical one | reading the regressions, the ablation | redesign the rewrite before trying again |
| one expected answer was too strict | reading the regressions | fix the eval set (9d does) |

On to **9d**: shipping it, and what production does next.